# GOES-R Series Introduction — Implementation / 구현

**Paper**: Goodman, S. J. (2019). "GOES-R Series Introduction." Chapter 1 in *The GOES-R Series: A New Generation of Geostationary Environmental Satellites* (Elsevier). DOI: 10.1016/B978-0-12-814327-8.00001-9.

This notebook implements three concrete pieces of physics from the GOES-R space-weather payload: (1) geostationary Earth-view geometry, (2) SUVI EUV image scaling (square-root and log stretches), and (3) EXIS X-ray response and flare classification.

이 노트북은 GOES-R 우주기상 탑재체의 세 가지 구체적 물리를 구현한다: (1) 정지궤도 지구 관측 기하, (2) SUVI EUV 영상 스케일링(제곱근/로그 스트레치), (3) EXIS X선 응답 및 플레어 등급 분류.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## Part 1: Geostationary Earth-View Geometry / 1부: 정지궤도 지구 관측 기하

GOES-R operates at $r_\mathrm{GEO} \approx 42{,}164$ km from Earth center. The full-disk angular diameter is $\theta_\mathrm{FD} = 2 \arcsin(R_\oplus / r_\mathrm{GEO}) \approx 17.4^\circ$. The ABI fixed-grid coordinate system uses $\pm 8.7^\circ$ as bounding box.

GOES-R은 지구 중심에서 $r_\mathrm{GEO} \approx 42{,}164$ km에서 운용된다. 풀디스크 각 직경은 $\theta_\mathrm{FD} = 2 \arcsin(R_\oplus / r_\mathrm{GEO}) \approx 17.4^\circ$이며, ABI fixed-grid 좌표계는 ±8.7°를 경계 박스로 사용한다.

In [ ]:
# Physical constants
GM_EARTH = 3.986004418e14  # m^3/s^2, Earth gravitational parameter
T_SIDEREAL = 86164.0905  # s, sidereal day
R_EARTH = 6378.137e3  # m, Earth equatorial radius


def compute_geo_radius(gm: float = GM_EARTH, period: float = T_SIDEREAL) -> float:
    """Compute geostationary orbital radius from Kepler's third law.

    Args:
        gm: Gravitational parameter of central body in m^3/s^2.
        period: Orbital period in seconds.

    Returns:
        Orbital radius from Earth center in meters.
    """
    return (gm * period ** 2 / (4 * np.pi ** 2)) ** (1.0 / 3.0)


def full_disk_angular_diameter(r_orbit: float, r_body: float = R_EARTH) -> float:
    """Compute full-disk angular diameter as seen from orbit.

    Args:
        r_orbit: Distance from body center in meters.
        r_body: Body radius in meters.

    Returns:
        Angular diameter in degrees.
    """
    return np.degrees(2.0 * np.arcsin(r_body / r_orbit))


r_geo = compute_geo_radius()
altitude = r_geo - R_EARTH
theta_fd = full_disk_angular_diameter(r_geo)

print(f"GEO radius:           {r_geo / 1e3:>12.3f} km")
print(f"Altitude above Earth: {altitude / 1e3:>12.3f} km")
print(f"Full-disk diameter:   {theta_fd:>12.3f} deg")
print(f"Half-angle (fixed-grid bound): {theta_fd / 2:.3f} deg")

In [ ]:
def slant_resolution_factor(phi_deg: float) -> float:
    """Compute the cos^-3 footprint enlargement factor at look angle phi.

    For a flat-Earth approximation in the limit of small IFOV,
    the ground footprint scales as 1/cos^3(phi) with respect to nadir.

    Args:
        phi_deg: Look angle from nadir in degrees.

    Returns:
        Multiplicative factor by which the nadir footprint enlarges.
    """
    phi_rad = np.radians(phi_deg)
    return 1.0 / np.cos(phi_rad) ** 3


phi = np.linspace(0, 70, 200)
factor = slant_resolution_factor(phi)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(phi, factor, lw=2)
ax.axhline(1, color='gray', lw=0.5, ls='--')
ax.set_xlabel('Look angle from nadir [deg] / 직하로부터의 관측각')
ax.set_ylabel('Footprint enlargement factor (1/cos³ φ) / 풋프린트 확대 계수')
ax.set_title('GOES-R ABI Off-Nadir Footprint Enlargement / 직하 밖 ABI 풋프린트 확대')
ax.set_yscale('log')
ax.grid(True, alpha=0.3, which='both')
ax.axvline(60, color='red', lw=0.5, ls=':', label='~60° useful imaging limit')
ax.legend()
plt.tight_layout()
plt.show()

for phi_test in [0, 30, 45, 60, 65, 70]:
    f = slant_resolution_factor(phi_test)
    print(f"phi = {phi_test:>3d} deg  =>  footprint x{f:>6.2f}")

In [ ]:
# Visualize Earth as seen from GEO (75.2 W, GOES-East)
fig, ax = plt.subplots(figsize=(7, 7))
earth = Circle((0, 0), 1.0, color='steelblue', alpha=0.4, label='Earth disk')
ax.add_patch(earth)

# Mark sub-satellite point and concentric look-angle rings (in radii fraction)
ax.plot(0, 0, 'k+', markersize=14, mew=2, label='Sub-satellite point\n75.2°W')
for phi_ring in [30, 45, 60]:
    # Project the ring of look-angle phi onto the visible disk
    r_proj = np.sin(np.radians(phi_ring)) / np.sin(np.radians(theta_fd / 2))
    if r_proj < 1:
        ring = Circle((0, 0), r_proj, fill=False, ls='--', color='darkred',
                      lw=0.8)
        ax.add_patch(ring)
        ax.text(0, r_proj, f'{phi_ring}°', color='darkred',
                ha='center', va='bottom', fontsize=9)

ax.set_xlim(-1.3, 1.3)
ax.set_ylim(-1.3, 1.3)
ax.set_aspect('equal')
ax.set_title('GOES-East Earth Disk (Schematic) / GOES-East 지구 디스크 (개념도)')
ax.legend(loc='lower left')
ax.axis('off')
plt.tight_layout()
plt.show()

## Part 2: SUVI EUV Image Scaling / 2부: SUVI EUV 영상 스케일링

SUVI observes the corona at six EUV wavelengths (94, 131, 171, 195, 284, 304 Å). The intensity dynamic range from coronal hole to flare core spans 4–5 orders of magnitude. To display these images on an 8-bit screen, a stretching transformation is used: square-root for moderate dynamic range, log for very high contrast.

SUVI는 6개 EUV 파장(94, 131, 171, 195, 284, 304 Å)에서 코로나를 관측한다. 코로나 홀에서 플레어 코어까지의 강도 동적 범위는 4–5 자릿수에 걸친다. 8-bit 화면에 표시하려면 스트레치 변환이 필요하다: 중간 동적 범위에는 제곱근, 고대비에는 로그.

In [ ]:
def make_synthetic_suvi(size: int = 256, seed: int = 42) -> np.ndarray:
    """Generate a synthetic SUVI-like coronal EUV image.

    The image consists of: a quiescent-corona background, several active
    regions (Gaussian peaks), one bright flare core, and a coronal hole
    (suppressed region). Intensities span ~5 orders of magnitude.

    Args:
        size: Image side length in pixels.
        seed: Random seed for reproducibility.

    Returns:
        2-D array of intensities (arbitrary DN units).
    """
    rng = np.random.default_rng(seed)
    y, x = np.mgrid[:size, :size]
    cx, cy = size / 2.0, size / 2.0

    # Disk mask (solar disk)
    r2 = (x - cx) ** 2 + (y - cy) ** 2
    disk = r2 < (size * 0.42) ** 2

    # Quiescent corona background (~1e3 DN on disk, low off-disk)
    img = np.where(disk, 1.0e3, 50.0)
    img = img + 50.0 * rng.standard_normal(img.shape)

    # Active regions
    for _ in range(4):
        ax_ = rng.uniform(0.3, 0.7) * size
        ay = rng.uniform(0.3, 0.7) * size
        sigma = rng.uniform(4, 10)
        amp = rng.uniform(5e3, 3e4)
        img += amp * np.exp(-((x - ax_) ** 2 + (y - ay) ** 2) / (2 * sigma ** 2))

    # Bright flare core
    fx, fy = 0.62 * size, 0.45 * size
    img += 5.0e5 * np.exp(-((x - fx) ** 2 + (y - fy) ** 2) / (2 * 3.5 ** 2))

    # Coronal hole
    hx, hy = 0.35 * size, 0.6 * size
    hole = np.exp(-((x - hx) ** 2 + (y - hy) ** 2) / (2 * 12.0 ** 2))
    img *= 1.0 - 0.95 * hole

    return np.clip(img, 1e-1, None)


def stretch_sqrt(img: np.ndarray,
                  vmin: float | None = None,
                  vmax: float | None = None) -> np.ndarray:
    """Apply square-root stretch and rescale to [0, 1].

    Args:
        img: Input image array.
        vmin: Lower clip; if None, taken as 1st percentile.
        vmax: Upper clip; if None, taken as 99.5th percentile.

    Returns:
        Stretched image in [0, 1].
    """
    if vmin is None:
        vmin = np.percentile(img, 1.0)
    if vmax is None:
        vmax = np.percentile(img, 99.5)
    clipped = np.clip(img, vmin, vmax)
    s = np.sqrt(clipped) - np.sqrt(vmin)
    s /= np.sqrt(vmax) - np.sqrt(vmin)
    return np.clip(s, 0.0, 1.0)


def stretch_log(img: np.ndarray,
                 vmin: float | None = None,
                 vmax: float | None = None) -> np.ndarray:
    """Apply natural-log stretch and rescale to [0, 1].

    Args:
        img: Input image array.
        vmin: Lower clip; if None, taken as 1st percentile.
        vmax: Upper clip; if None, taken as 99.9th percentile.

    Returns:
        Stretched image in [0, 1].
    """
    if vmin is None:
        vmin = max(np.percentile(img, 1.0), 1e-6)
    if vmax is None:
        vmax = np.percentile(img, 99.9)
    clipped = np.clip(img, vmin, vmax)
    s = np.log(clipped) - np.log(vmin)
    s /= np.log(vmax) - np.log(vmin)
    return np.clip(s, 0.0, 1.0)


suvi = make_synthetic_suvi(size=256)
print(f"Synthetic SUVI image shape: {suvi.shape}")
print(f"Min: {suvi.min():.2e}, Max: {suvi.max():.2e}, Dynamic range: {suvi.max() / suvi.min():.2e}")

In [ ]:
# Compare three display strategies: linear, square-root, log
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

linear_disp = np.clip(suvi / suvi.max(), 0, 1)
sqrt_disp = stretch_sqrt(suvi)
log_disp = stretch_log(suvi)

for ax, disp, title in zip(axes,
                            [linear_disp, sqrt_disp, log_disp],
                            ['Linear / 선형',
                             'Square-root / 제곱근',
                             'Log / 로그']):
    ax.imshow(disp, cmap='hot', origin='lower')
    ax.set_title(title)
    ax.axis('off')

fig.suptitle('SUVI EUV Display Stretches / SUVI EUV 디스플레이 스트레치')
plt.tight_layout()
plt.show()

In [ ]:
# Histogram of intensities, before and after stretching
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(np.log10(suvi.flatten()), bins=80, color='steelblue')
axes[0].set_title('log10(I) histogram (raw) / 원본')
axes[0].set_xlabel('log10(intensity)')

axes[1].hist(sqrt_disp.flatten(), bins=80, color='orange')
axes[1].set_title('After sqrt stretch / 제곱근 후')
axes[1].set_xlabel('display value [0,1]')

axes[2].hist(log_disp.flatten(), bins=80, color='crimson')
axes[2].set_title('After log stretch / 로그 후')
axes[2].set_xlabel('display value [0,1]')

for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part 3: EXIS X-Ray Response and Flare Classification / 3부: EXIS X선 응답 및 플레어 등급

The EXIS XRS instrument measures solar X-ray flux in two bands: XRS-A (0.5–4 Å) and XRS-B (1–8 Å, the canonical GOES flare-classification band). The flare class is determined by the peak XRS-B flux: A < 10⁻⁷, B 10⁻⁷–10⁻⁶, C 10⁻⁶–10⁻⁵, M 10⁻⁵–10⁻⁴, X ≥ 10⁻⁴ W/m².

EXIS XRS 기기는 두 밴드에서 태양 X선 플럭스를 측정한다: XRS-A(0.5–4 Å), XRS-B(1–8 Å, 고전적 GOES 플레어 분류 밴드). 플레어 등급은 최고 XRS-B 플럭스에 의해 결정된다.

In [ ]:
def classify_flare(flux_w_m2: float) -> str:
    """Convert peak GOES XRS-B flux to flare class string (e.g., 'M5.0').

    The class letter is set by the order of magnitude:
        A: F < 1e-7,    B: 1e-7..1e-6,    C: 1e-6..1e-5,
        M: 1e-5..1e-4,  X: F >= 1e-4 W/m^2.
    The mantissa (rounded to one decimal) is the number after the letter,
    capped at 9.9 below the next class boundary.

    Args:
        flux_w_m2: Peak 1-8 angstrom flux in W/m^2.

    Returns:
        Flare class string such as 'C3.4' or 'X9.3'.
    """
    if flux_w_m2 < 1e-8:
        return 'sub-A'
    if flux_w_m2 < 1e-7:
        letter, base = 'A', 1e-8
    elif flux_w_m2 < 1e-6:
        letter, base = 'B', 1e-7
    elif flux_w_m2 < 1e-5:
        letter, base = 'C', 1e-6
    elif flux_w_m2 < 1e-4:
        letter, base = 'M', 1e-5
    else:
        letter, base = 'X', 1e-4
    mantissa = flux_w_m2 / base
    return f'{letter}{mantissa:.1f}'


# Verify with historic events
events = [
    ("Feb 2014 background", 1.5e-8),
    ("B-class flare", 5e-7),
    ("Mid C flare", 3.4e-6),
    ("M5 flare", 5e-5),
    ("X1 flare", 1.0e-4),
    ("X9.3 (Sept 2017)", 9.3e-4),
    ("Carrington estimate", 4.5e-3),
]
for label, flux in events:
    print(f"{label:>22s}  F = {flux:.2e} W/m^2  =>  {classify_flare(flux)}")

In [ ]:
def synthetic_flare_lightcurve(t: np.ndarray,
                                t_peak: float,
                                rise: float,
                                decay: float,
                                peak_flux: float,
                                background: float = 1e-7) -> np.ndarray:
    """Generate a synthetic flare light curve (rise + exponential decay).

    A widely used simple model: exponential rise to peak followed by
    exponential decay. Useful for testing flare-detection logic.

    Args:
        t: Time array (minutes).
        t_peak: Peak time (minutes).
        rise: 1/e rise time (minutes).
        decay: 1/e decay time (minutes).
        peak_flux: Peak flux above background in W/m^2.
        background: Constant background flux in W/m^2.

    Returns:
        Flux time series in W/m^2.
    """
    flux = np.full_like(t, background, dtype=float)
    rising = t < t_peak
    decaying = ~rising
    flux[rising] += peak_flux * np.exp((t[rising] - t_peak) / rise)
    flux[decaying] += peak_flux * np.exp(-(t[decaying] - t_peak) / decay)
    return flux


t_min = np.linspace(0, 120, 1200)  # 2 hours, ~6-second cadence
lightcurve = synthetic_flare_lightcurve(
    t_min, t_peak=45.0, rise=4.0, decay=15.0,
    peak_flux=9.3e-4, background=2e-7,
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(t_min, lightcurve, lw=1.5, color='black')

# Add classification bands
for level, label, color in [
    (1e-8, 'A', 'lightgray'),
    (1e-7, 'B', 'lightblue'),
    (1e-6, 'C', 'palegreen'),
    (1e-5, 'M', 'gold'),
    (1e-4, 'X', 'salmon'),
]:
    ax.axhline(level, color=color, lw=0.8, ls='--')
    ax.text(0.5, level * 1.3, label, color='dimgray',
            fontsize=9, fontweight='bold')

peak_idx = lightcurve.argmax()
ax.plot(t_min[peak_idx], lightcurve[peak_idx], 'r*', markersize=18,
        label=f'Peak: {classify_flare(lightcurve[peak_idx])}')
ax.set_xlabel('Time [min]')
ax.set_ylabel('XRS-B flux [W/m^2] (1-8 A)')
ax.set_title('Synthetic GOES EXIS XRS-B Flare Light Curve / 합성 GOES EXIS XRS-B 플레어 광도곡선')
ax.set_ylim(1e-9, 5e-3)
ax.grid(True, which='both', alpha=0.3)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
def integrate_band_response(
    wavelength: np.ndarray,
    spectrum: np.ndarray,
    response: np.ndarray,
) -> float:
    """Compute band-integrated flux given wavelength-dependent response.

    Implements F_band = integral( R(lambda) * Phi(lambda) ) d lambda.

    Args:
        wavelength: Wavelength grid in angstroms.
        spectrum: Spectral flux density Phi(lambda) [W/m^2/A].
        response: Dimensionless wavelength response function R(lambda).

    Returns:
        Band-integrated flux in W/m^2.
    """
    return float(np.trapz(response * spectrum, wavelength))


# Construct an idealized XRS-A and XRS-B band response (top-hat)
lam = np.linspace(0.1, 12, 1200)  # angstroms
resp_a = np.where((lam >= 0.5) & (lam <= 4.0), 1.0, 0.0)
resp_b = np.where((lam >= 1.0) & (lam <= 8.0), 1.0, 0.0)

# Construct a synthetic flare spectrum: hot bremsstrahlung continuum.
# Phi(lambda) ~ exp(-hc/(lambda kT)) / lambda^2, scaled so peak XRS-B ~ 1e-4 W/m^2.
kT_keV = 1.5  # keV (approx isothermal flare core temperature: T ~ 17 MK)
hc_kev_A = 12.398
energy_keV = hc_kev_A / lam
spectrum = np.exp(-energy_keV / kT_keV) / lam ** 2
spectrum *= 1e-4 / spectrum.max()  # normalize peak

f_a = integrate_band_response(lam, spectrum, resp_a)
f_b = integrate_band_response(lam, spectrum, resp_b)

print(f"XRS-A (0.5-4 A) integrated flux: {f_a:.3e} W/m^2")
print(f"XRS-B (1-8 A) integrated flux:  {f_b:.3e} W/m^2  =>  class {classify_flare(f_b)}")
print(f"Hardness ratio XRS-A/XRS-B:      {f_a / f_b:.3f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.semilogy(lam, spectrum, color='black', lw=1.2)
ax1.set_xlabel('Wavelength [A]')
ax1.set_ylabel('Phi(lambda) [W/m^2/A]')
ax1.set_title('Synthetic flare X-ray spectrum (kT ~ 1.5 keV) / 합성 플레어 X선 스펙트럼')
ax1.grid(True, alpha=0.3)

ax2.plot(lam, resp_a, label='XRS-A (0.5-4 A)', lw=1.5)
ax2.plot(lam, resp_b, label='XRS-B (1-8 A)', lw=1.5)
ax2.set_xlabel('Wavelength [A]')
ax2.set_ylabel('Response R(lambda)')
ax2.set_title('Idealized EXIS XRS Band Responses / EXIS XRS 밴드 응답 (이상화)')
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary / 요약

| Concept / 개념 | This Paper / 본 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| GEO altitude / 정지궤도 고도 | $r_\mathrm{GEO} \approx 35,786$ km above Earth | All operational GOES; replicated by every GEO mission |
| Full-disk angular diameter | $\theta_\mathrm{FD} \approx 17.4^\circ$ | ABI fixed-grid $\pm 8.7^\circ$; analogous to MTG-FCI, AHI |
| Off-nadir resolution / 비직하 해상도 | $1/\cos^3\phi$ degradation | All GEO imagers; limit of useful imaging ≈60°–65° |
| EUV stretches / EUV 스트레치 | sqrt and log mappings | SDO/AIA helioviewer.org; SOLAR-ORBITER/EUI |
| X-ray flare class / X선 플레어 등급 | A/B/C/M/X based on XRS-B 1–8 Å flux | Continuous from GOES-1 (1975) to today's EXIS |
| Band-integrated response | $F_\mathrm{band} = \int R(\lambda) \Phi(\lambda) d\lambda$ | All XRS instruments; calibration against synchrotron sources |

**Takeaway / 시사점** — Even though Goodman's 3-page chapter is a program-level orientation, the physics it implies (orbital mechanics, instrument geometry, image dynamic range, X-ray classification) is concrete and computable. This notebook codifies those relations as reusable Python utilities suitable for analyzing real GOES-R L1b/L2 NetCDF granules.

Goodman의 3페이지 장은 프로그램 레벨 소개이지만, 그 안에 함축된 물리(궤도 역학, 기기 기하, 영상 동적 범위, X선 등급)는 구체적이며 계산 가능하다. 본 노트북은 이러한 관계를 실제 GOES-R L1b/L2 NetCDF 그래뉼 분석에 재사용 가능한 Python 유틸리티로 정리한다.